# About this notebook
- Load trained model from checkpoint
- Do inference on test set
- Visualize results
- Compute Metrics

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os

# Get the absolute path of the current directory
current_dir = os.getcwd()

# Get the parent directory (project root)
# If your notebook is deeper (e.g., notebooks/exploratory/), you might need os.path.dirname() twice
project_root = os.path.dirname(os.path.dirname(current_dir))

# Add project root to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)

# Verify it points to the folder containing 'src'
print(f"Project Root added: {project_root}")

# Setup

In [ ]:
import os
import random
import torch
import torch.nn.functional as F
import logging
import matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import src.utils.utils as utils
import src.tasks.taskloader as taskloader
import src.data.dataloader as dataloader
import src.train.training as training

from typing import List, Tuple, Dict, Optional
from torch.utils.data import DataLoader
from torchvision.transforms.functional import to_pil_image, pil_to_tensor
from sklearn.metrics import f1_score, jaccard_score

from src.data.dataloader import get_metadata_df
from src.utils.utils import setup_logging
from src.dataset.general_dataset import NeuralizerGeneralDataset
from src.dataset.sampler import WeightedTaskSampler
from src.tasks.base_task import BaseTask
from src.neuralizer.lightning_model import LightningModel
from src.visualizations.contextvizualizer import plot_context_set, plot_context_set_overlay

setup_logging(loglevel=logging.INFO)
logger = logging.getLogger(__name__)

# Set logging level of external libraries to INFO to avoid cluttering
logging.getLogger('PIL').setLevel(logging.INFO)
logging.getLogger('matplotlib').setLevel(logging.INFO)

os.chdir("../..")
print(os.getcwd())

# Load CFG file
CFG: dict = utils.load_config("configs/config.yaml")

# Matplotlib settings
matplotlib.rcParams['mathtext.fontset'] = 'cm'  
matplotlib.rcParams['font.family'] = 'STIXGeneral'

# Params of this Notebook
Specify the params of this notebook here

In [ ]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True

# Version
PATH_TO_CHECKPOINT = '/path/to/your/checkpoint.ckpt'  # <-- UPDATE THIS PATH


# Set to True if you want to use CPU for inference, otherwise 'cuda:x' will be used
USE_CPU = False

# Specifiy cuda device
GPU_IDX = 0
CUDA_DEVICE = f"cuda:{GPU_IDX}"
device = torch.device("cpu" if USE_CPU else CUDA_DEVICE)
print(f"Cuda is available: {torch.cuda.is_available()}\nUsing device: '{device}' for Inference ...")

# Load trained Model

In [ ]:
if "neuralizer_base" in PATH_TO_CHECKPOINT:
    model = LightningModel.load_from_checkpoint(
        PATH_TO_CHECKPOINT,
        learning_rate=CFG["training"]["model_instantiation"]["learning_rate"],
        batch_size=CFG["training"]["dataloader"]["batch_size"],
        strict=False,
    )
else:
    model: LightningModel = training.load_model(
        CFG=CFG, path_to_checkpoint=PATH_TO_CHECKPOINT, cuda_device=CUDA_DEVICE, use_cpu=USE_CPU
    )

In [ ]:
# Load metadata
dataloader.load_data(CFG)
metadata_df: pd.DataFrame = dataloader.get_metadata_df(CFG["dataloader"]["metadata_location"])

# Get tasks
tasks: List[BaseTask] = taskloader.get_tasks(df=metadata_df, CFG=CFG["tasks"])

# Get test tasks (tasks unseen during training)
test_tasks: List[BaseTask] = taskloader.get_test_tasks(
    df=metadata_df, CFG=CFG["tasks"]
)

# Get train, validation and test set as DataLoader objects
train_dataloader, val_dataloader, test_dataloader = training.get_dataloaders(CFG=CFG, tasks=tasks, test_mode=True)

_, _, test_tasks_dataloader = training.get_dataloaders(CFG=CFG, tasks=test_tasks, test_mode=True)


# Sample a random batch from the test dataloader

In [ ]:
# Combine your task lists
tasks_combined = tasks + test_tasks

# Print a header for clarity
print(f"{'Index':<10} | {'Task Name/Object'}")
print("-" * 40)

# Iterate through the list to map every index
for idx, task in enumerate(tasks_combined):
    print(f"{idx:<10} | {task}")

# Show the total count to avoid future IndexErrors
print("-" * 40)
print(f"Total tasks available: {len(tasks_combined)}")

In [ ]:
# Speficiy which task you want to plot the test images
# Set it to None to allow all tasks during inference
_TASK_INDEX: Optional[int] = 2  # 0-DENOISING,1-DERAINING,2-SEGMENTATION,3-INPAINTING,4-SUPERRESOLUTION,5-DEPTHESTIMATION and 6-EDGEDETECTION respectively

tasks_combined = tasks + test_tasks

if _TASK_INDEX is not None:
    task_of_interest = tasks_combined[_TASK_INDEX]
    _, _, test_dataloader = training.get_dataloaders(CFG=CFG, tasks=[task_of_interest], test_mode=True)

# Get batch from test dataloader
x, y, ctx_in, ctx_out, lossfun, class_names  = next(iter(test_dataloader))

print("x shape: ", x.shape)
print("y shape: ", y.shape)
print("ctx_in shape: ", ctx_in.shape)
print("ctx_out shape: ", ctx_out.shape)
print("loss function(s): ", lossfun)
print("task name: ", class_names)

assert x.ndim == 4 and y.ndim == 4
assert ctx_in.ndim == 5 and ctx_out.ndim == 5
assert x.dtype == torch.float32 and x.max() <= 1 and x.min() >= 0
assert y.dtype == torch.float32 and y.max() <= 1  and y.min() >= 0
assert ctx_in.dtype == torch.float32 and ctx_in.max() <= 1  and ctx_in.min() >= 0
assert ctx_out.dtype == torch.float32 and ctx_out.max() <= 1  and ctx_out.min() >= 0

# Make Prediction

In [ ]:
# Put data and model to device
model = model.to(device)
x = x.to(device)
ctx_in = ctx_in.to(device)
ctx_out = ctx_out.to(device)

# Do inference
model.eval()
with torch.no_grad():
    y_pred = model(x, ctx_in, ctx_out)

assert y_pred.dtype == torch.float32 and y_pred.max() <= 1 and y_pred.min() >= 0

# Visualize Prediction

In [ ]:

from src.visualizations.prediction_visualizer import plot_prediction_batch, plot_single_prediction

if len(test_dataloader.dataset.tasks) == 1:
    suffix_fig = test_dataloader.dataset.tasks[0].get_task_name()

plot_prediction_batch(x, y, y_pred, class_names, print_metrics=True)